In [1]:

import pandas as pd
from pathlib import Path

# Project data folder
DATA_DIR = Path.cwd().parent / "data"

customers = pd.read_csv(DATA_DIR / "customers.csv")
employees = pd.read_csv(DATA_DIR / "employees.csv")
services = pd.read_csv(DATA_DIR / "services.csv")
bookings = pd.read_csv(DATA_DIR / "bookings.csv")
job_assignments = pd.read_csv(DATA_DIR / "job_assignments.csv")
expenses = pd.read_csv(DATA_DIR / "expenses.csv")

datasets = {
    "Customers": customers,
    "Employees": employees,
    "Services": services,
    "Bookings": bookings,
    "Job Assignments": job_assignments,
    "Expenses": expenses
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows, {len(df.columns)} columns")


Customers: 750 rows, 4 columns
Employees: 25 rows, 4 columns
Services: 4 rows, 4 columns
Bookings: 18,653 rows, 6 columns
Job Assignments: 28,480 rows, 6 columns
Expenses: 216 rows, 4 columns


In [2]:

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())



--- Customers ---
Missing values: 0
Duplicate rows: 0

--- Employees ---
Missing values: 0
Duplicate rows: 0

--- Services ---
Missing values: 0
Duplicate rows: 0

--- Bookings ---
Missing values: 0
Duplicate rows: 0

--- Job Assignments ---
Missing values: 0
Duplicate rows: 0

--- Expenses ---
Missing values: 0
Duplicate rows: 0


In [3]:

# Convert date columns
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
employees["hire_date"] = pd.to_datetime(employees["hire_date"])
bookings["booking_date"] = pd.to_datetime(bookings["booking_date"])

# Check bookings before customer signup
booking_check = bookings.merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left"
)

invalid_bookings = booking_check[
    booking_check["booking_date"] < booking_check["signup_date"]
]

print("Bookings before customer signup:", len(invalid_bookings))

# Check assignments before employee hire
assignment_check = (
    job_assignments
    .merge(
        bookings[["booking_id", "booking_date"]],
        on="booking_id",
        how="left"
    )
    .merge(
        employees[["employee_id", "hire_date"]],
        on="employee_id",
        how="left"
    )
)

invalid_assignments = assignment_check[
    assignment_check["booking_date"] < assignment_check["hire_date"]
]

print("Assignments before employee hire:", len(invalid_assignments))


Bookings before customer signup: 0
Assignments before employee hire: 0


In [4]:

# ---------------------------------
# CUSTOMER BOOKING FREQUENCY ANALYSIS
# ---------------------------------

completed = bookings[
    bookings["status"] == "Completed"
].copy()

# Count completed bookings per customer
booking_counts = (
    completed.groupby("customer_id")
    .size()
    .reset_index(name="completed_bookings")
)

# Include customers with zero completed bookings
customer_activity = customers.merge(
    booking_counts,
    on="customer_id",
    how="left"
)

customer_activity["completed_bookings"] = (
    customer_activity["completed_bookings"]
    .fillna(0)
    .astype(int)
)

# Summarize activity by cleaning frequency
frequency_summary = (
    customer_activity
    .groupby("cleaning_frequency")
    .agg(
        customers=("customer_id", "count"),
        avg_completed_bookings=("completed_bookings", "mean"),
        total_completed_bookings=("completed_bookings", "sum")
    )
    .round(2)
)

print(frequency_summary)


                    customers  avg_completed_bookings  \
cleaning_frequency                                      
Biweekly                  296                   25.25   
Monthly                   134                   12.40   
One-time                  161                    0.93   
Weekly                    159                   49.89   

                    total_completed_bookings  
cleaning_frequency                            
Biweekly                                7475  
Monthly                                 1662  
One-time                                 150  
Weekly                                  7933  


In [5]:

# ---------------------------------
# FINAL DATA QUALITY CHECKS
# ---------------------------------

# Verify unique booking IDs
duplicate_booking_ids = bookings["booking_id"].duplicated().sum()

# Verify cancelled jobs have no revenue
cancelled_with_revenue = bookings[
    (bookings["status"] == "Cancelled") &
    (bookings["price"] != 0)
]

# Verify no assignments exist for cancelled jobs
cancelled_assignments = job_assignments.merge(
    bookings[["booking_id", "status"]],
    on="booking_id",
    how="left"
)

cancelled_assignments = cancelled_assignments[
    cancelled_assignments["status"] == "Cancelled"
]

# Verify customer references exist
missing_customers = (
    ~bookings["customer_id"].isin(customers["customer_id"])
).sum()

# Verify employee references exist
missing_employees = (
    ~job_assignments["employee_id"].isin(
        employees["employee_id"]
    )
).sum()

print("Duplicate booking IDs:", duplicate_booking_ids)
print("Cancelled bookings with revenue:", len(cancelled_with_revenue))
print("Assignments on cancelled bookings:", len(cancelled_assignments))
print("Bookings with missing customers:", missing_customers)
print("Assignments with missing employees:", missing_employees)



Duplicate booking IDs: 0
Cancelled bookings with revenue: 0
Assignments on cancelled bookings: 0
Bookings with missing customers: 0
Assignments with missing employees: 0
